# 06 — Lineage

**What you will learn:**
- Edge sources: `ViewLineage` (parsed from view SQL), `QueryLineage` (parsed from the query log), `Manual` (UI/API), plus connector-specific edges (dashboards, pipelines, dbt).
- The lineage API: nodes + upstream/downstream edges, depth, column lineage (`columnsLineage` with `fromColumns`/`toColumn`).
- Adding lineage by API (`PUT /v1/lineage`) with column mappings and a description.
- Impact analysis: everything downstream of a column or table, to a chosen depth, as a table you can send to owners.
- The limits: what the SQL parser misses, query-log windows, stale edges.

This chapter is the lineage stop of Level 3 of the tutorial tree ("each job in depth", see `plan.md`). It deepens the *trace* job from chapter 01, question 3: *where do lineage edges come from, how far can I trust them, and how do I use them to answer "what breaks if I change this"?* It builds on chapters 01 (lineage seen once), 04 (the lineage workflow, the query log, workflow order) and 05 (PUT create-or-update, entity references, ids vs FQNs).

**Quick grasp.** A lineage edge is one fact: asset A feeds asset B ("from → to"). *Upstream* of a table means its sources; *downstream* means its consumers. OpenMetadata draws edges from three places: view definitions it can parse, the database query log (for tables built by `CREATE TABLE AS`, like our nightly job), and people or programs that add edges by hand through the UI or API for everything the connectors cannot see (a dashboard tool with no connector, a Python job). Impact analysis is a downstream walk: start at the thing you want to change, list everything reachable, and tell the owners. That is the whole chapter: read the edges, add the missing one, walk the graph.

In [1]:
import sys
from pathlib import Path

here = Path.cwd()
ROOT = next(d for d in [here, *here.parents, here / "project"] if (d / "pyproject.toml").exists() and (d / "src" / "openmetadata_tutorial").exists())
sys.path.insert(0, str(ROOT / "src"))

from openmetadata_tutorial import nb
ROOT = nb.setup()

import importlib
from openmetadata_tutorial import client, ch06_lineage
ch06_lineage = importlib.reload(ch06_lineage)
print("project root:", ROOT)

project root: /Users/sergii/.ai/knowledge/tutorials/openmetadata/project


This zooms into the first piece of the picture — reading edges — and answers: *what does one lineage edge look like, and where did it come from?* We start with the smallest possible case: one edge, upstream of `analytics.daily_revenue`, depth 1.

In [2]:
edges_1 = ch06_lineage.edges(ch06_lineage.DAILY_REVENUE, up=1, down=0)
edges_1

,from,to,source
0,brightcup_pg.brightcup.analytics.fct_orders,brightcup_pg.brightcup.analytics.daily_revenue,ViewLineage


One row, one edge: `analytics.fct_orders → analytics.daily_revenue`, source `ViewLineage`. That source value is the answer to "where did it come from": `ViewLineage` means OpenMetadata's lineage workflow read the view's SQL and parsed the `FROM` clause. No query log involved, no human involved. **What it means for us:** every edge carries its own provenance in the `source` column — read it before trusting the edge.

In [3]:
print(ch06_lineage.view_sql(ch06_lineage.DAILY_REVENUE))

create view analytics.daily_revenue as  SELECT order_date,
    count(*) AS orders,
    sum(total_amount) AS revenue
   FROM analytics.fct_orders
  GROUP BY order_date;


That is the exact SQL the parser read (`GET /v1/tables/name/...daily_revenue?fields=schemaDefinition` — note the field is called `schemaDefinition`, not `viewDefinition`, for views in this version). One `FROM analytics.fct_orders`, one edge: the parser did the obvious correct thing. **What it means for us:** when an edge looks wrong, this SQL is the first place to look — lineage bugs start in the definition, not in the graph.

Same question one level wider: *what does the whole Brightcup graph look like?* The cell below unions two walks (everything upstream of `customer_ltv`, plus everything upstream of `daily_revenue` — no single center reaches all five edges, because the two branches only meet at `fct_orders`).

In [4]:
graph = ch06_lineage.brightcup_graph()
graph

,from,to,source
0,brightcup_pg.brightcup.analytics.fct_orders,brightcup_pg.brightcup.analytics.customer_ltv,QueryLineage
1,brightcup_pg.brightcup.analytics.fct_orders,brightcup_pg.brightcup.analytics.daily_revenue,ViewLineage
2,brightcup_pg.brightcup.raw.customers,brightcup_pg.brightcup.analytics.customer_ltv,QueryLineage
3,brightcup_pg.brightcup.raw.customers,brightcup_pg.brightcup.analytics.fct_orders,ViewLineage
4,brightcup_pg.brightcup.raw.orders,brightcup_pg.brightcup.analytics.fct_orders,ViewLineage


In [5]:
print(ch06_lineage.to_mermaid(graph))

flowchart LR
    n0["fct_orders"]
    n1["customers"]
    n2["orders"]
    n3["customer_ltv"]
    n4["daily_revenue"]
    n0 -->|Query| n3
    n0 -->|View| n4
    n1 -->|Query| n3
    n1 -->|View| n0
    n2 -->|View| n0


```mermaid
flowchart LR
    n0["fct_orders"]
    n1["customers"]
    n2["orders"]
    n3["customer_ltv"]
    n4["daily_revenue"]
    n0 -->|Query| n3
    n0 -->|View| n4
    n1 -->|Query| n3
    n1 -->|View| n0
    n2 -->|View| n0
```

Five edges, four tables, two sources. Read it left to right: `raw.orders` and `raw.customers` both feed `fct_orders` (view SQL — the join in `03_views.sql`); `fct_orders` feeds `daily_revenue` (view SQL again); and both `raw.customers` and `fct_orders` feed `customer_ltv` — but with source `QueryLineage`, because `customer_ltv` is a *table*, not a view: there is no view SQL to parse, so only the query log could link it. **What it means for us:** the `source` column tells you which machinery stands behind each edge, and each kind has different failure modes (next sections).

> **Important:** Lineage is only as complete as what produced it.
> **What it means for us:** Views and logged queries are parsed; Python/Spark jobs, BI tools without a connector, and file exports are invisible until a connector or a manual edge adds them. An empty downstream does not mean "safe to change" — it may mean "nobody told the catalog".
> **Why it matters:** The graph above has no dashboard and no Python job, yet the finance dashboard reads `daily_revenue` every night. Trusting the graph as-is would miss the most important consumer.

This zooms into the edge itself and answers: *which column feeds which?* Table-level lineage says "something downstream breaks"; column lineage says *what* breaks. First the mappings on the `fct_orders → daily_revenue` edge:

In [6]:
ch06_lineage.column_edges(ch06_lineage.DAILY_REVENUE, up=1, down=0)

,from,to,from_column,to_column,source
0,brightcup_pg.brightcup.analytics.fct_orders,brightcup_pg.brightcup.analytics.daily_revenue,order_date,order_date,ViewLineage
1,brightcup_pg.brightcup.analytics.fct_orders,brightcup_pg.brightcup.analytics.daily_revenue,total_amount,revenue,ViewLineage


Two mappings: `order_date → order_date`, and `fct_orders.total_amount → daily_revenue.revenue` (note the rename — the mapping survives it, which is exactly the point). Now follow one raw column end to end:

In [7]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
ch06_lineage.column_path(ch06_lineage.AMOUNT_COLUMN)

,step,from_column,to_column
0,1,brightcup_pg.brightcup.raw.orders.total_amount,brightcup_pg.brightcup.analytics.fct_orders.total_amount
1,2,brightcup_pg.brightcup.analytics.fct_orders.total_amount,brightcup_pg.brightcup.analytics.daily_revenue.revenue
2,2,brightcup_pg.brightcup.analytics.fct_orders.total_amount,brightcup_pg.brightcup.analytics.customer_ltv.lifetime_value


Two steps, three rows. Step 1: `raw.orders.total_amount` feeds `fct_orders.total_amount`. Step 2 forks: `fct_orders.total_amount` feeds `daily_revenue.revenue` *and* `customer_ltv.lifetime_value` — the two money metrics. One column, two business metrics. **What it means for us:** this fork is why column lineage, not table lineage, drives the impact analysis later: renaming `total_amount` hits two metrics, and only the column walk shows both.

> **Important:** Column lineage is what makes impact analysis useful.
> **What it means for us:** Table-level lineage says "something downstream"; column-level says "this metric" — here `revenue` and `lifetime_value`, the two metrics a rename of `total_amount` would break.
> **Why it matters:** Renames and type changes are column-level events: without this walk you would know *something* breaks but not which metrics.

This zooms into the `QueryLineage` rows and answers: *where did those come from — there is no view SQL for `customer_ltv`?* They came from the query log. `customer_ltv` is rebuilt nightly by `just transform` (`CREATE TABLE AS ... FROM raw.customers ... JOIN analytics.fct_orders`), which lands in Postgres's `pg_stat_statements`; the `postgres-lineage` workflow reads recent statements (`queryLogDuration: 1` day in `ingestion/lineage.yaml`) and parses them. The cell below shows both sides: the logged statement itself, searched from the server's query log (`GET /v1/queries`), and the nightly-job definition from the local job file (which reads the same on every run — the logged copy can age out of the log window):

In [8]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
log = ch06_lineage.query_log_total()
print("statements in the server log:", log["total"])
qs = ch06_lineage.queries_for(ch06_lineage.CUSTOMER_LTV)
print("statements mentioning customer_ltv:", len(qs))
for _, row in qs.iterrows():
    print("--- logged statement ---")
    print(row["query"])
print("--- nightly-job definition (transform.py) ---")
print(ch06_lineage.nightly_job_sql())

statements in the server log: 34
statements mentioning customer_ltv: 2
--- logged statement ---
CREATE TABLE analytics.customer_ltv AS
SELECT c.customer_id,
       c.email,
       c.country,
       count(f.order_id)               AS orders,
       coalesce(sum(f.total_amount), $1) AS lifetime_value
FROM raw.customers c
LEFT JOIN analytics.fct_orders f ON f.customer_id = c.customer_id
GROUP BY c.customer_id, c.email, c.country
--- logged statement ---
SELECT count(*) FROM analytics.customer_ltv
--- nightly-job definition (transform.py) ---
CREATE TABLE analytics.customer_ltv AS
SELECT c.customer_id,
       c.email,
       c.country,
       count(f.order_id)               AS orders,
       coalesce(sum(f.total_amount), 0) AS lifetime_value
FROM raw.customers c
LEFT JOIN analytics.fct_orders f ON f.customer_id = c.customer_id
GROUP BY c.customer_id, c.email, c.country


The two blocks agree almost to the character: the logged statement is the nightly job's `CREATE TABLE AS` (`FROM raw.customers ... LEFT JOIN analytics.fct_orders`, `GROUP BY ...`), except `coalesce(sum(f.total_amount), 0)` arrives logged as `coalesce(sum(f.total_amount), $1)` — `pg_stat_statements` normalises literals to `$n`. That `$1` is the evidence the first block is the server's copy, not our file. The two `QueryLineage` edges are this one statement, parsed while it was still inside the log window: its two inputs (`raw.customers`, `fct_orders`) are the two edges' sources. (The second logged statement, `SELECT count(*) FROM analytics.customer_ltv`, is a read — it produces no lineage.) One honesty note: the `entityFQN` filter on `GET /v1/queries` is ignored by the server in 2.0.5 (observed: any FQN returns the same first page as the unfiltered list), so the helper lists the log and searches it locally. **What it means for us:** query lineage exists only for queries the log still holds when the workflow runs.

> **Important:** Query lineage depends on the query-log window.
> **What it means for us:** Run the lineage workflow regularly; a nightly job that falls outside `queryLogDuration` is missed, and its edges silently never appear (or disappear on fresh deployments).
> **Why it matters:** If `just transform` ran a week ago and the workflow reads one day of log, `customer_ltv` shows *no* upstream — looking exactly like an ungoverned manual table. The edges above survived because the workflow ran while the statement was still in the window.

This answers the gap the first callout named: *what about lineage no connector can see?* The finance team's dashboard tool is not connected to OpenMetadata, so nothing will ever draw the `daily_revenue → dashboard` edge for us. We register a stand-in: a dashboard service, one chart ("Revenue by day"), one dashboard ("Finance overview"). All three calls are PUT create-or-update, so re-running keeps the same FQNs. Note the service type: `CustomDashboard`, not Metabase — there is no real dashboard server to point at, and a Metabase type with a dummy URL would pretend a connection exists.

In [9]:
ch06_lineage.ensure_dashboard()

{'service': 'brightcup_metabase',
 'chart': 'brightcup_metabase.revenue_by_day',
 'dashboard': 'brightcup_metabase.finance_overview',
 'dashboard_id': '041b272f-63ca-4f1f-af69-e1fbafac8dad',
 'how': 'PUT create-or-update (re-running keeps the same FQNs)'}

Service `brightcup_metabase`, chart `brightcup_metabase.revenue_by_day`, dashboard `brightcup_metabase.finance_overview`. Now the edge, `PUT /v1/lineage`: table → dashboard carries the description on the edge itself — `lineageDetails` is valid only table → table (the server answers 400 otherwise), so dashboard edges use this form.

In [10]:
ch06_lineage.add_edge(
    ch06_lineage.DAILY_REVENUE, "table",
    "brightcup_metabase.finance_overview", "dashboard",
    "Finance overview reads daily_revenue nightly",
)

{'from': 'brightcup_pg.brightcup.analytics.daily_revenue',
 'to': 'brightcup_metabase.finance_overview',
 'description': 'Finance overview reads daily_revenue nightly',
 'how': 'manual edge'}

Re-read downstream of `raw.orders` (depth 3 — the server maximum; depth 4 answers 400 "must be less than or equal to 3", so the helper clamps):

In [11]:
down = ch06_lineage.edges(ch06_lineage.ORDERS, up=1, down=3)
down

,from,to,source
0,brightcup_pg.brightcup.raw.orders,brightcup_pg.brightcup.analytics.fct_orders,ViewLineage
1,brightcup_pg.brightcup.analytics.fct_orders,brightcup_pg.brightcup.analytics.daily_revenue,ViewLineage
2,brightcup_pg.brightcup.analytics.fct_orders,brightcup_pg.brightcup.analytics.customer_ltv,QueryLineage
3,brightcup_pg.brightcup.analytics.daily_revenue,brightcup_metabase.finance_overview,Manual


Four rows. The chain now ends at the dashboard: `daily_revenue → brightcup_metabase.finance_overview`, source `Manual` — the one edge in the table no workflow could have drawn. The UI shows the same chain — the lineage view of `daily_revenue`, with `fct_orders` upstream and `Finance overview` downstream:

![](assets/06_lineage_with_dashboard.png)

(The header also shows owner `Finance`, domain `Sales`, tier and certification — chapter 08's governance work, already in the catalog; lineage and governance share the same page.)

> **Important:** Manual edges do not update themselves.
> **What it means for us:** Give every manual edge a description (and later, an owner) and review them: this one says the dashboard reads the table *nightly*, so a reader can judge whether it still holds.
> **Why it matters:** If the finance team stops using the table, the edge stays and the next impact analysis names a consumer that no longer exists — a false alarm, or worse, a false sense of coverage.

All pieces are on the board; now the question the chapter promised: *we will rename `raw.orders.total_amount` to `amount_eur` — what breaks?* First the precise, column-level walk (depth 3, the server maximum — the helper clamps deeper requests, as the re-read above showed):

In [12]:
col_impact = ch06_lineage.impact(ch06_lineage.ORDERS, column=ch06_lineage.AMOUNT_COLUMN)
col_impact

,level,kind,fqn,owners,via_column
0,1,table,brightcup_pg.brightcup.analytics.fct_orders,,total_amount
1,2,table,brightcup_pg.brightcup.analytics.daily_revenue,Finance,revenue
2,2,table,brightcup_pg.brightcup.analytics.customer_ltv,,lifetime_value


Three rows. Level 1: `fct_orders` (via `total_amount`, no owner yet — chapter 08 adds owners). Level 2: `daily_revenue` (via `revenue`, owned by team `Finance`) and `customer_ltv` (via `lifetime_value`, no owner yet). The dashboard is *not* in this table: manual dashboard edges carry no column mapping, so the column walk stops at `daily_revenue`. That is not a gap in the method, it is the method telling the truth about its resolution. The table-level walk goes one step further:

In [13]:
tab_impact = ch06_lineage.impact(ch06_lineage.ORDERS)
tab_impact[["level", "kind", "fqn", "owners"]]

,level,kind,fqn,owners
0,1,table,brightcup_pg.brightcup.analytics.fct_orders,
1,2,table,brightcup_pg.brightcup.analytics.daily_revenue,Finance
2,2,table,brightcup_pg.brightcup.analytics.customer_ltv,
3,3,dashboard,brightcup_metabase.finance_overview,


Four rows, ending at level 3, kind `dashboard`: `brightcup_metabase.finance_overview` — no owner, no `via_column` (same reason as above). So the plain-words answer to send to the team: renaming `total_amount` touches `fct_orders.total_amount`, which feeds two metrics — `daily_revenue.revenue` (owned by Finance, shown on the Finance overview dashboard) and `customer_ltv.lifetime_value` (no owner yet). Tell Finance, and give `customer_ltv` an owner before the rename. And the quality angle in one sentence, with the evidence below: the currently failing test on the source flows to the same dashboard —

In [14]:
ch06_lineage.quality_status(ch06_lineage.ORDERS)

,test,status
0,orders_no_negative_amounts,Failed
1,orders_row_count_between,Success


The table above names the failing test; the test itself names a number. Count the offending rows directly in the shop database:

In [15]:
ch06_lineage.negative_amount_count()

{'rows': 5, 'how': 'live SQL against the shop database'}

`orders_no_negative_amounts` is Failed — and the live count agrees: 5 negative-amount rows sit in `raw.orders` right now (chapter 07 runs these tests properly). Every euro it flags already flowed into `revenue` and `lifetime_value`. One note on the numbers: the plan's designer run saw three failures on this table, but right now the catalog holds two test cases on `raw.orders` (one Failed, one Success), so *one* failing test flows to the dashboard today. Impact analysis plus one failing test is the whole "can I trust the dashboard?" story in miniature.

Last level: *how far can I trust the parser?* This section creates a new view — `analytics.ch06_dynamic`, built to be awkward: CTEs, `UNION ALL` over two tables, a function call (`upper`), an expression (`quantity * unit_price`) — created here for the first time in this chapter as `shop_admin` (idempotent `CREATE OR REPLACE`). The earlier walks in this chapter exclude it by default, which is why they read the same before and after it exists; the reads below opt back in with `exclude=[]`:

In [16]:
ch06_lineage.create_dynamic_view()

{'view': 'brightcup_pg.brightcup.analytics.ch06_dynamic',
 'how': 'CREATE OR REPLACE (re-running keeps the same view)'}

Re-run the two shared workflows (idempotent) so the catalog picks the view up, then read what the parser made of it. Each run prints its progress from inside the ingestion container and ends with a `Workflow Success %: 100.0` summary plus its runtime — the `tail -2` lines below keep just those summaries:

In [17]:
!just ingest metadata 2>&1 | tail -2
!just ingest lineage 2>&1 | tail -2

[2026-10-12 13:06:59] INFO     {metadata.Utils:logger:223} - Workflow Success %: 100.0
[2026-10-12 13:06:59] INFO     {metadata.Utils:logger:223} - Workflow finished in time: 517.397ms


[2026-10-12 13:07:03] INFO     {metadata.Utils:logger:223} - Workflow Success %: 100.0
[2026-10-12 13:07:03] INFO     {metadata.Utils:logger:223} - Workflow finished in time: 431.034ms


In [18]:
dyn = ch06_lineage.edges(ch06_lineage.DYNAMIC_VIEW, up=3, down=0, exclude=[])
dyn

,from,to,source
0,brightcup_pg.brightcup.raw.order_items,brightcup_pg.brightcup.analytics.ch06_dynamic,ViewLineage
1,brightcup_pg.brightcup.raw.orders,brightcup_pg.brightcup.analytics.ch06_dynamic,ViewLineage


Two table edges — but the column-mapping claim needs its own output. Read the column mappings on those two edges:

In [19]:
ch06_lineage.column_edges(ch06_lineage.DYNAMIC_VIEW, up=3, down=0, exclude=[])

,from,to,from_column,to_column,source
0,brightcup_pg.brightcup.raw.order_items,brightcup_pg.brightcup.analytics.ch06_dynamic,quantity,amount,ViewLineage
1,brightcup_pg.brightcup.raw.order_items,brightcup_pg.brightcup.analytics.ch06_dynamic,unit_price,amount,ViewLineage
2,brightcup_pg.brightcup.raw.order_items,brightcup_pg.brightcup.analytics.ch06_dynamic,order_id,order_id,ViewLineage
3,brightcup_pg.brightcup.raw.orders,brightcup_pg.brightcup.analytics.ch06_dynamic,total_amount,amount,ViewLineage
4,brightcup_pg.brightcup.raw.orders,brightcup_pg.brightcup.analytics.ch06_dynamic,customer_id,customer_id,ViewLineage
5,brightcup_pg.brightcup.raw.orders,brightcup_pg.brightcup.analytics.ch06_dynamic,order_id,order_id,ViewLineage
6,brightcup_pg.brightcup.raw.orders,brightcup_pg.brightcup.analytics.ch06_dynamic,status,status,ViewLineage


The honest result, for this one view: the parser resolved it *fully* — the table above shows both table edges (`raw.orders`, `raw.order_items`), and this one shows 7 column mappings, including through the CTE, both `UNION ALL` branches (`total_amount` and `quantity`/`unit_price` both map to `amount`), and the `upper(status)` call, which arrives as a plain `status → status` mapping (the parser followed the function to its argument). That is a result about this one view, not about dynamic SQL in general: one view, one dialect (Postgres), one run. Not tested here: SQL built as a string inside a Python job, or a statement that aged out of the query-log window — both stay invisible, for the reasons the callouts above gave. Two staleness rules to close with: edges parsed from views refresh on every lineage run (change the view, re-run, the edge follows); manual edges never refresh — they stay until a person deletes them. The view stays in the catalog afterwards on purpose: dropping it would soft-delete it on the next metadata run and confuse later chapters.

## Carry forward

- Three edge sources: **view SQL, query log, manual/API** (+ connectors of other systems) — and every edge says which one it is.
- The lineage API returns **nodes + edges with column mappings**; depth is your choice, up to the server maximum of 3.
- **Impact analysis = downstream walk + owners**, and it is only as good as the coverage: the dashboard shows up only because we added the edge by hand.